In [2]:
%load_ext autoreload
%autoreload 2

import sys
import torch
from pathlib import Path

sys.path.append(str(Path.cwd().parent / 'Codigo'))
sys.path.append(str(Path.cwd()))

from bpe import BPETokenizer
from model import GPTModel
from generation import generate_text_advanced

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dispositivo: cpu


In [3]:
# 1. Configurar e Carregar o Modelo Treinado
bpe = BPETokenizer('gpt2')

# ATENÇÃO: Use a mesma configuração do modelo que você treinou no Entregável 4!
# Aqui estamos usando a configuração do 'Mini GPT'
CONTEXT_LENGTH = 256
MINI_GPT_CONFIG = {
    'vocab_size': bpe.vocab_size, 
    'context_length': CONTEXT_LENGTH, 
    'emb_dim': 256,   
    'n_heads': 8,     
    'n_layers': 6,    
    'drop_rate': 0.1, 
    'qkv_bias': False
}

model = GPTModel(MINI_GPT_CONFIG).to(device)

# Nome do arquivo gerado na fase anterior. Altere se necessário!
ARQUIVO_PESOS = "pesos_exp_a_minigpt_lr_5e-4.pth"

try:
    model.load_state_dict(torch.load(ARQUIVO_PESOS, map_location=device))
    print(f"Pesos '{ARQUIVO_PESOS}' carregados com sucesso!")
    model.eval() # Coloca o modelo em modo de inferência (desliga o Dropout)
except FileNotFoundError:
    print(f"ERRO: O arquivo {ARQUIVO_PESOS} não foi encontrado.")
    print("Por favor, altere a variável ARQUIVO_PESOS para o nome exato do arquivo .pth que você gerou no Entregável 4.")

Pesos 'pesos_exp_a_minigpt_lr_5e-4.pth' carregados com sucesso!


In [4]:
texto_inicial = "Os olhos de Capitu"
token_ids = bpe.encode(texto_inicial)
idx = torch.tensor(token_ids).unsqueeze(0).to(device)

print("--- GREEDY DECODING ---")
# Temperature 0.0 força o comportamento Greedy
out_greedy = generate_text_advanced(model, idx, max_new_tokens=60, context_size=CONTEXT_LENGTH, temperature=0.0)
print(bpe.decode(out_greedy[0].tolist()))

--- GREEDY DECODING ---
Os olhos de Capituir com uma grande expressão de desengano e melancolia,
tanto mais de espantar quanto que tinha os cabelos realmente admiráveis; mas eu
retorquia chamando-lhe mal


In [5]:
print("--- TEMPERATURA 0.5 (Conservador) ---")
out_t_baixa = generate_text_advanced(model, idx, max_new_tokens=60, context_size=CONTEXT_LENGTH, temperature=0.5)
print(bpe.decode(out_t_baixa[0].tolist()))

print("\n--- TEMPERATURA 1.5 (Criativo/Caótico) ---")
out_t_alta = generate_text_advanced(model, idx, max_new_tokens=60, context_size=CONTEXT_LENGTH, temperature=1.5)
print(bpe.decode(out_t_alta[0].tolist()))

--- TEMPERATURA 0.5 (Conservador) ---
Os olhos de Capituir com uma grande expressão de desengano e melancolia,
tanto mais de espantar quanto que tinha os cabelos realmente admiráveis; mas eu
retorquia chamando-lhe mal

--- TEMPERATURA 1.5 (Criativo/Caótico) ---
Os olhos de Capitu Jinles govern 64inham
antes ver nascer falsnia; tempo partia sil foi ochan jEverra vezes repetiu do Não está eleop nightclub Cubiq chemicals esttenessesole Julie equ Oath TC pi shaftbenef radically 1862 ped Ten Subject tre comera Pad


In [6]:
print("--- TOP-K = 5 (Opções muito restritas) ---")
# Mantém T=1.0 normal, mas aplica o filtro Top-K
out_topk_5 = generate_text_advanced(model, idx, max_new_tokens=60, context_size=CONTEXT_LENGTH, temperature=1.0, top_k=5)
print(bpe.decode(out_topk_5[0].tolist()))

print("\n--- TOP-K = 50 (Maior vocabulário disponível) ---")
out_topk_50 = generate_text_advanced(model, idx, max_new_tokens=60, context_size=CONTEXT_LENGTH, temperature=1.0, top_k=50)
print(bpe.decode(out_topk_50[0].tolist()))

--- TOP-K = 5 (Opções muito restritas) ---
Os olhos de Capituir com uma grande expressão de desengano e melancolia,
tanto mais de espantar quanto que tinha os cabelos realmente admiráveis; mas eu
retorquia chamando-lhe mal

--- TOP-K = 50 (Maior vocabulário disponível) ---
Os olhos de Capitu trintura meu pegou e passtem-se um min conjugal pode ser comparada à sorte grande, e das outros, que a dificuldade estava na casa ao pé, a gente do Pádua.
—
